In [1]:
# ============================================
# Project 12 — Feature Engineering Performance Lab
# Notebook 2 — Feature Engineering
# Goal: Engineer 10+ new features from raw Telco data
# Strategy: Domain, Ratio, Interaction, Binning
# Prajwal Kondala | IIT KGP → AI/ML Engineer
# ============================================

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

# Same preprocessing as baseline!
df = df.drop('customerID', axis=1)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna()
df['Churn'] = (df['Churn'] == 'Yes').astype(int)

print("Shape:", df.shape)
print("Ready for feature engineering! ✅")

Shape: (7032, 20)
Ready for feature engineering! ✅


In [3]:
# ============================================
# Ratio Features
# ============================================

# Feature 1: Charges per month of tenure
# New customer paying high charges = higher risk!
df['ChargesPerTenure'] = df['MonthlyCharges'] / (df['tenure'] + 1)

# Feature 2: Total to Monthly ratio
# How many months worth have they paid?
df['TotalToMonthly'] = df['TotalCharges'] / (df['MonthlyCharges'] + 1)

print("Ratio features created! ✅")
print(df[['ChargesPerTenure', 'TotalToMonthly']].describe())

Ratio features created! ✅
       ChargesPerTenure  TotalToMonthly
count       7032.000000     7032.000000
mean           5.714882       31.793095
std            8.567435       24.148511
min            0.264384        0.949495
25%            1.250000        8.606988
50%            2.073598       28.228590
75%            5.884842       54.309968
max           51.225000       75.518072


In [4]:
# ============================================
# Domain Features
# ============================================

# Feature 3: Is new customer — danger zone!
df['IsNewCustomer'] = (df['tenure'] <= 6).astype(int)

# Feature 4: Is long term customer — loyal zone!
df['IsLongTermCustomer'] = (df['tenure'] >= 24).astype(int)

# Feature 5: Is auto payment — committed customer!
df['IsAutoPayment'] = df['PaymentMethod'].isin(
    ['Credit card (automatic)', 'Bank transfer (automatic)']
).astype(int)

# Feature 6: Service count — how engaged is the customer?
service_cols = ['PhoneService', 'OnlineSecurity', 'OnlineBackup',
                'DeviceProtection', 'TechSupport',
                'StreamingTV', 'StreamingMovies']

df['ServiceCount'] = df[service_cols].apply(
    lambda x: (x == 'Yes').sum(), axis=1
)

print("Domain features created! ✅")
print("\nIsNewCustomer distribution:")
print(df['IsNewCustomer'].value_counts())
print("\nServiceCount distribution:")
print(df['ServiceCount'].describe())

Domain features created! ✅

IsNewCustomer distribution:
IsNewCustomer
0    5562
1    1470
Name: count, dtype: int64

ServiceCount distribution:
count    7032.000000
mean        2.941411
std         1.843695
min         0.000000
25%         1.000000
50%         3.000000
75%         4.000000
max         7.000000
Name: ServiceCount, dtype: float64


In [5]:
# ============================================
# Interaction Features
# ============================================

# Feature 7: Fiber optic + Month-to-month = highest risk combo!
df['FiberAndMonthly'] = (
    (df['InternetService'] == 'Fiber optic') &
    (df['Contract'] == 'Month-to-month')
).astype(int)

# Feature 8: No online security + Fiber = vulnerable + expensive!
df['FiberAndNoSecurity'] = (
    (df['InternetService'] == 'Fiber optic') &
    (df['OnlineSecurity'] == 'No')
).astype(int)

# Feature 9: New customer + Month-to-month = double danger!
df['NewAndMonthly'] = (
    (df['tenure'] <= 6) &
    (df['Contract'] == 'Month-to-month')
).astype(int)

print("Interaction features created! ✅")
print("\nFiberAndMonthly:", df['FiberAndMonthly'].sum(), "customers")
print("FiberAndNoSecurity:", df['FiberAndNoSecurity'].sum(), "customers")
print("NewAndMonthly:", df['NewAndMonthly'].sum(), "customers")

Interaction features created! ✅

FiberAndMonthly: 2128 customers
FiberAndNoSecurity: 2257 customers
NewAndMonthly: 1413 customers


In [6]:
# ============================================
# Binning Feature
# ============================================

# Feature 10: Tenure groups — captures non-linear tenure effect!
df['TenureGroup'] = pd.cut(
    df['tenure'],
    bins=[0, 6, 12, 24, 72],
    labels=['New', 'Early', 'Mid', 'Loyal']
)

print("Binning feature created! ✅")
print("\nTenureGroup distribution:")
print(df['TenureGroup'].value_counts().sort_index())
print("\nChurn rate by TenureGroup:")
print(df.groupby('TenureGroup', observed=True)['Churn'].mean().round(3))

Binning feature created! ✅

TenureGroup distribution:
TenureGroup
New      1470
Early     705
Mid      1024
Loyal    3833
Name: count, dtype: int64

Churn rate by TenureGroup:
TenureGroup
New      0.533
Early    0.359
Mid      0.287
Loyal    0.140
Name: Churn, dtype: float64


In [7]:
# ============================================
# Encode — Engineer first, encode after!
# ============================================

df_engineered = pd.get_dummies(df, drop_first=True)

print("Shape after encoding:", df_engineered.shape)
print("New features added:", df_engineered.shape[1] - 31)

Shape after encoding: (7032, 43)
New features added: 12


In [8]:
# ============================================
# Sanity Check — before we proceed!
# ============================================

# Check for nulls
print("Null values:", df_engineered.isnull().sum().sum())

# Check for infinity
print("Infinity values:", df_engineered.isin([np.inf, -np.inf]).sum().sum())

# Confirm new features are there
new_features = ['ChargesPerTenure', 'TotalToMonthly',
                'IsNewCustomer', 'IsLongTermCustomer',
                'IsAutoPayment', 'ServiceCount',
                'FiberAndMonthly', 'FiberAndNoSecurity',
                'NewAndMonthly']

print("\nNew features present:")
for f in new_features:
    print(f" ✅ {f}")

Null values: 0
Infinity values: 0

New features present:
 ✅ ChargesPerTenure
 ✅ TotalToMonthly
 ✅ IsNewCustomer
 ✅ IsLongTermCustomer
 ✅ IsAutoPayment
 ✅ ServiceCount
 ✅ FiberAndMonthly
 ✅ FiberAndNoSecurity
 ✅ NewAndMonthly


In [10]:
# ============================================
# Additional Features — Behavior Driven!
# ============================================

# Feature 11: Contract Risk Score
# Month-to-month = highest risk, Two year = lowest!
contract_risk_map = {
    'Month-to-month' : 3,
    'One year'       : 2,
    'Two year'       : 1
}
df['ContractRiskScore'] = df['Contract'].map(contract_risk_map)

print("ContractRiskScore distribution:")
print(df['ContractRiskScore'].value_counts())
print("\nChurn rate by ContractRiskScore:")
print(df.groupby('ContractRiskScore')['Churn'].mean().round(3))

ContractRiskScore distribution:
ContractRiskScore
3    3875
1    1685
2    1472
Name: count, dtype: int64

Churn rate by ContractRiskScore:
ContractRiskScore
1    0.028
2    0.113
3    0.427
Name: Churn, dtype: float64


In [11]:
# Feature 12: Lifetime Value Approximation
# Expensive + loyal vs Expensive + risky!
df['LifetimeValueApprox'] = df['MonthlyCharges'] * df['tenure']

print("\nLifetimeValueApprox stats:")
print(df['LifetimeValueApprox'].describe().round(2))
print("\nAvg LifetimeValue by Churn:")
print(df.groupby('Churn')['LifetimeValueApprox'].mean().round(2))


LifetimeValueApprox stats:
count    7032.00
mean     2283.15
std      2264.70
min        18.80
25%       397.80
50%      1394.57
75%      3791.25
max      8550.00
Name: LifetimeValueApprox, dtype: float64

Avg LifetimeValue by Churn:
Churn
0    2555.20
1    1531.61
Name: LifetimeValueApprox, dtype: float64


In [12]:
# Feature 13: Price Shock Feature
# High charges before loyalty forms = danger!
median_charges = df['MonthlyCharges'].median()

df['PriceShockFeature'] = (
    (df['MonthlyCharges'] > median_charges) &
    (df['tenure'] <= 6)
).astype(int)

print("Median MonthlyCharges:", median_charges)
print("\nPriceShockFeature distribution:")
print(df['PriceShockFeature'].value_counts())
print("\nChurn rate by PriceShockFeature:")
print(df.groupby('PriceShockFeature')['Churn'].mean().round(3))

Median MonthlyCharges: 70.35

PriceShockFeature distribution:
PriceShockFeature
0    6505
1     527
Name: count, dtype: int64

Churn rate by PriceShockFeature:
PriceShockFeature
0    0.228
1    0.736
Name: Churn, dtype: float64


In [13]:
# Sanity check — all new features clean!
new_features = ['ContractRiskScore', 'LifetimeValueApprox',
                'PriceShockFeature']

print("Null check:")
print(df[new_features].isnull().sum())
print("\nFeature ranges:")
print(df[new_features].describe().round(2))

Null check:
ContractRiskScore      0
LifetimeValueApprox    0
PriceShockFeature      0
dtype: int64

Feature ranges:
       ContractRiskScore  LifetimeValueApprox  PriceShockFeature
count            7032.00              7032.00            7032.00
mean                2.31              2283.15               0.07
std                 0.83              2264.70               0.26
min                 1.00                18.80               0.00
25%                 2.00               397.80               0.00
50%                 3.00              1394.57               0.00
75%                 3.00              3791.25               0.00
max                 3.00              8550.00               1.00
